# Vulkan Vacuum Grasping Lab
### *Adaptive In-Orbit Servicing & Geometric Intelligence*
---
## Pipeline Overview
1.  **Ingest** complex STL/PLY/PCD geometries.
2.  **Normalize** data (Unit scaling & Normal estimation).
3.  **Align** the object based on solid topological surfaces (RANSAC + DBSCAN).
4.  **Simulate** vacuum seal physics using the **Grasp Stability Score (GSS)**.
5.  **Identify** optimal contact points for robotic servicing.

## 🛠️ 1. Environment Setup
Initialize core geometric utilities and custom grasping modules.

In [1]:
import os
from os.path import join as pjoin
import numpy as np
import open3d as o3d
import yaml
from copy import deepcopy
import matplotlib.pyplot as plt

# Custom Framework Modules
from src.grippers.vacuum_gripper_v2 import VacuumGripper, VacuumGripperConfig
from src.grasping.vacuum_sampler_v2 import VacuumGraspSampler, VacuumSamplerConfig
import src.utils.geometry_utils as gu
from src.generic_geometry import GenericGeometry

print("✅ Framework Ready.")

✅ Framework Ready.


## 2. Selection Library
Configure your experiment by selecting a target part and a suction cup array. This block replaces messy path strings with an organized collection.

In [12]:
# === [COLLECTION] Gripper Parameters ===
GRIPPER_COLLECTION = {
    "single_small": "single_circle_cup_1cm_diam.yaml",
    "single_large": "single_circle_cup_2cm_diam.yaml",
    "double_standard": "double_cup_1cm_1cm.yaml",
    "double_large": "double_cup.yaml",
    "quad_array": "quadruple_cup_1cm_1cm.yaml",
    "franka_vacuum": "franka_vacuum.yaml"
}

# === [COLLECTION] Vulkan Part Library ===
VULKAN_PARTS = {
    "part_1": r"Test_part/Vulkan/stl/Vulkan part 1.stl",
    "part_2": r"Test_part/Vulkan/stl/Vulkan part 2.stl",
    "part_3": r"Test_part/Vulkan/stl/Vulkan part 3.stl",
    "part_4_body": r"Test_part/Vulkan/stl/Vulkan part 4-Body.stl",
    "nut_m12": r"Test_part/Vulkan/stl/Vulkan M12 Nut.stl",
    "subassembly": r"Test_part/Vulkan/stl/Vulkan subassembly JMS paper.stl",
    "subassembly_new": r"Test_part/Vulkan/stl/Vulkan subassembly JMS paper.stl",
    "complete_assembly": r"Test_part/Vulkan/stl/Subassembly.ply",
    "screw": r"Test_part/Vulkan/stl/Vulkan screw.stl"
}

# --- CHANGE YOUR SELECTION HERE ---
selected_gripper = "single_small"
selected_part = "subassembly"
# ----------------------------------

gripper_path = pjoin("gripper_parameter", GRIPPER_COLLECTION[selected_gripper])
object_path = VULKAN_PARTS[selected_part]

gripper = VacuumGripper(gripper_path)
pcd = GenericGeometry(object_path)

print(f" Gripper Initialized: {gripper.config.name}")
print(f" Object Loaded: {os.path.basename(object_path)}")

✅ Geometry set: mesh (open3d)
Applying rigid transformation...
🔧 Vacuum Gripper 'Double_Schmalz_ECG' initialized.
   - Active Pads: 1
[Open3D] Loaded Mesh: Test_part/Vulkan/stl/Vulkan subassembly JMS paper.stl
✅ Geometry set: mesh (open3d)
 Gripper Initialized: Double_Schmalz_ECG
 Object Loaded: Vulkan subassembly JMS paper.stl


##  3. Pre-Visualization
Check the initial spatial relationship. Use this to verify the object was loaded correctly.

In [13]:
print("Opening 3D Preview...")
o3d.visualization.draw_geometries(
    [gripper.collision_geometry.geometry, pcd.geometry], 
    window_name="Initial Setup Preview",
    point_show_normal=True
)

Opening 3D Preview...


## 4. Geometric Normalization
Convert to **meters**, downsample to a workable resolution, and remove outliers. 
> **Voxel Size**: defines the 'grain' of the surface.

In [14]:
report = pcd.get_dimensions_report()
report

{'extents_xyz': array([ 70.9752, 101.6867,  73.1604]),
 'diagonal': 143.9796,
 'likely_unit': 'millimeters',
 'suggested_voxel_size': 1.4398,
 'details': 'Type: Open3D Mesh. Size: 70.98 x 101.69 x 73.16. Unit: MILLIMETERS.'}

#### Manual Evaluation:
The report gives a heuristic/notion if the scale is in millimeters or meters. If the "likely_unit" is "millimeters", we apply a scaling factor of 0.001 to convert to meters. This is crucial for ensuring that all subsequent geometric computations are accurate and consistent with the gripper's dimensions, which are typically defined in meters.

#### **However** the object may already be in meters, so it's important to make a manual check here

In [15]:
# ONLY RUN IF YOU WANT TO SCALE IT 1000X SMALLER (MM -> M for ex)
pcd.scale(0.001)
report = pcd.get_dimensions_report()
report

{'extents_xyz': array([0.071 , 0.1017, 0.0732]),
 'diagonal': 0.144,
 'likely_unit': 'meters',
 'suggested_voxel_size': 0.00144,
 'details': 'Type: Open3D Mesh. Size: 0.07 x 0.10 x 0.07. Unit: METERS.'}

I normally used the suggested voxel size of the dimension report rounded down (or a standard value like 0.0005).

In [16]:
# 2. Intelligent Downsampling
voxel_size = report["suggested_voxel_size"]/2 
pcd_down = pcd.downsample(voxel_size=voxel_size)
pcd_down = GenericGeometry(geometry=pcd_down)

# 3. Noise Reduction
# (OPTIONAL)
# pcd_down.remove_outliers_physical(inplace=True, min_neighbors=5)

print("--- Preprocessing Complete ---")
print(pcd_down.get_dimensions_report()['details'])
pcd_down.visualize()

Input is Mesh. Sampling surface before downsampling...
Downsampling with voxel_size=0.00072...
✅ Geometry set: point_cloud (open3d)
--- Preprocessing Complete ---
Type: Open3D PointCloud. Size: 0.07 x 0.10 x 0.07. Unit: METERS.


##  5. Intelligent Plane Alignment
We use **Hybrid RANSAC + DBSCAN** to find the top K solid surfaces. 
### **NOTE**: This should "ideally" be used for files which aren't already oriented (like the vulkan ones). But the cube-sat, for example, should run without this step to respect the normal z-axis (crucial for verticality evaluation).
- `k=3`: Generates 3 candidate part orientations.
- `theta_min_diff=30`: Ensures orientations are significantly different from each other.

In [17]:
pcd_aligned_list = gu.align_largest_plane_to_z(
    pcd_down.geometry, 
    distance_threshold=1.5 * voxel_size, 
    k=3, 
    theta_min_diff=30.0
)

print(f"Successfully identified {len(pcd_aligned_list)} primary alignment planes.")

for i, geom in enumerate(pcd_aligned_list):
    print(f"Viewing Aligned Geometry {i+1}...")
    GenericGeometry(geometry=geom).visualize()

Successfully identified 3 primary alignment planes.
Viewing Aligned Geometry 1...
✅ Geometry set: point_cloud (open3d)
Viewing Aligned Geometry 2...
✅ Geometry set: point_cloud (open3d)
Viewing Aligned Geometry 3...
✅ Geometry set: point_cloud (open3d)


In [ ]:
# If you decide not to ru the above cell, run this one for compatibility with the rest of the workflow
pcd_aligned_list = [pcd_down.geometry]

## 6. Grasp Sampling Engine
Initialize the sampler using `config.yaml` and execute the discovery pipeline across all identified orientations.

In [ ]:
# Load Sampling Parameters
config_path = pjoin("config", "config.yaml")
with open(config_path, 'r') as f:
    raw_config = yaml.safe_load(f)

sampler_config = VacuumSamplerConfig(**raw_config)
sampler = VacuumGraspSampler(gripper, sampler_config)

all_candidates = []

print("Starting Batch Sampling...")
for i, aligned_geom in enumerate(pcd_aligned_list):
    print(f"--- Processing Plane {i+1}/{len(pcd_aligned_list)} ---")
    found_grasps = sampler.sample_grasps(aligned_geom)
    all_candidates.extend(found_grasps)

# Global Ranking
all_candidates.sort(key=lambda x: x.score, reverse=True)
print(f"\n✨ Discovery Complete: Found {len(all_candidates)} valid candidates.")

## 📊 7. Visualization & Analytics
Gain insights into the grasp quality using spatial heatmaps and detailed pad projections.

In [ ]:
# === [ANALYSIS] Score Heatmap ===
# Attributes: 'total', 'seal_score', 'torque', 'verticality'
sampler.visualize_candidates_heatmap(pcd_down.geometry, attribute="total", relative_scale=True)

In [ ]:
# === [ANALYSIS] Winner Inspection ===
if all_candidates:
    top_grasp = all_candidates[0]
    
    print(f"🏆 BEST GRASP DETAILS")
    print(f"Global Score: {top_grasp.score:.4f}")
    print("----------------------")
    for key, val in top_grasp.score_details.items():
        if key != 'pad_scores':
            print(f"{key.replace('_',' ').title()}: {val}")
            
    # Visualize with 'Collision Shield' enabled (Blue cylinders showing safety volume)
    sampler.visualize_grasp(pcd_down.geometry, top_grasp, show_safety_volume=True)
else:
    print("❌ No grasps found. Consider lowering 'min_score' or increasing 'max_angle_deg' in config.yaml")

## 🔍 8. Deep Physics Debugging
Visualize exactly how the suction pads are projected onto the object surface for specific candidates. This explains why a grasp might have a low Sealing Score.

In [ ]:
# Select a candidate by index (0 is the best)
CANDIDATE_TO_DEBUG = 0

if CANDIDATE_TO_DEBUG < len(all_candidates):
    target = all_candidates[CANDIDATE_TO_DEBUG]
    print(f"🔍 Inspecting Candidate #{CANDIDATE_TO_DEBUG}...")
    sampler.debug_specific_grasp(pcd_down.geometry, target)
else:
    print("Index out of range.")